# Deep CFR advantage learning-rate schedules

This experiment tests whether reducing the advantage-network learning rate after the early learning phase improves the learned-average plateau. All variants receive the same measured neural-training time. Exact evaluations run asynchronously and are excluded from that budget.

Only the advantage optimizers are scheduled. The smaller `(512, 512)` average-strategy network and its `1e-3` learning rate remain fixed.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import json
import math
import os
import shutil
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.algo.deep_cfr import DeepCFRTrainer
from liars_poker.core import GameSpec
from liars_poker.eval.deep_cfr_async import AsyncDeepCFREvaluator
from liars_poker.serialization import save_policy

assert torch.cuda.is_available(), 'This experiment requires CUDA.'
device = torch.device('cuda')
torch.set_float32_matmul_precision('high')
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('free / total GPU GiB:', tuple(round(x / 2**30, 2) for x in torch.cuda.mem_get_info()))

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

MINUTES_PER_VARIANT = 30
EVALUATE_EVERY_MINUTES = 5
TRAVERSALS_PER_PLAYER = 1024
SEED = 7

TRAINER_KWARGS = {
    'advantage_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'device': device,
    'seed': SEED,
    'advantage_buffer_capacity': 2_000_000,
    'strategy_buffer_capacity': 2_000_000,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'advantage_train_steps': 100,
    'strategy_train_steps': 50,
    'advantage_positive_weight': 0.5,
    'strategy_weighting': 'linear',
    'highest_regret_fallback': True,
    'alternating_updates': True,
    'validation_fraction': 0.02,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native',
    'traversal_batch_size': 1024,
    'device_replay': True,
    'fused_optimizer': True,
    'amp_dtype': None,
    'compile_models': False,
}

def constant_lr(measured_s, total_s):
    return 1e-3

def step_lr(measured_s, total_s):
    minutes = measured_s / 60.0
    if minutes < 10.0:
        return 1e-3
    if minutes < 20.0:
        return 3e-4
    return 1e-4

def cosine_lr(measured_s, total_s):
    progress = min(max(measured_s / total_s, 0.0), 1.0)
    maximum, minimum = 1e-3, 1e-4
    return minimum + 0.5 * (maximum - minimum) * (1.0 + math.cos(math.pi * progress))

SCHEDULES = {
    'constant 1e-3': constant_lr,
    'step 1e-3→3e-4→1e-4': step_lr,
    'cosine 1e-3→1e-4': cosine_lr,
}

experiment_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
EXPERIMENT_DIR = (
    REPO_ROOT / 'artifacts' / 'deep_cfr_lr_scheduler'
    / f'{spec.to_short_str()}___{experiment_id}'
)
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
print('Experiment directory:', EXPERIMENT_DIR)

In [ ]:
def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f'Not JSON serializable: {type(value)!r}')

def append_jsonl(path, records):
    if not records:
        return
    with path.open('a', encoding='utf-8') as handle:
        for record in records:
            handle.write(json.dumps(record, default=json_default) + '\n')

def load_jsonl(path):
    if not path.exists():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]

def set_advantage_lr(trainer, learning_rate):
    for optimizer in trainer.advantage_optimizers:
        for group in optimizer.param_groups:
            group['lr'] = float(learning_rate)

def collect_and_cleanup(evaluator, results_path, *, wait=False):
    results = evaluator.wait() if wait else evaluator.collect_ready()
    append_jsonl(results_path, results)
    for result in results:
        shutil.rmtree(Path(result['policy_dir']), ignore_errors=True)
    return results

def normalized_auc(frame):
    frame = frame.sort_values('measured_training_s')
    if len(frame) < 2:
        return float('nan')
    x = frame['measured_training_s'].to_numpy(float)
    y = frame['exploitability'].to_numpy(float)
    return float(np.trapezoid(y, x) / (x[-1] - x[0]))

## Run equal-time schedules

No exact-resume checkpoints are stored. Each variant keeps only logs, exact-evaluation results, and its final playable average policy.

In [ ]:
RUN_EXPERIMENT = True
runs = {}
total_training_s = MINUTES_PER_VARIANT * 60.0

if RUN_EXPERIMENT:
    for label, schedule in SCHEDULES.items():
        print(f'\n=== {label} ===')
        safe_label = (
            label.replace(' ', '_').replace('→', '_to_').replace('/', '_')
        )
        run_dir = EXPERIMENT_DIR / safe_label
        run_dir.mkdir(parents=True, exist_ok=True)
        training_path = run_dir / 'training.jsonl'
        results_path = run_dir / 'exact_results.jsonl'
        policy_dir = run_dir / 'policy'

        trainer = DeepCFRTrainer(spec, **TRAINER_KWARGS)
        evaluator = AsyncDeepCFREvaluator(
            run_dir,
            max_workers=1,
            compile_batch_size=65_536,
        )
        measured_s = 0.0
        next_evaluation_s = EVALUATE_EVERY_MINUTES * 60.0

        try:
            while measured_s < total_training_s:
                advantage_lr = schedule(measured_s, total_training_s)
                set_advantage_lr(trainer, advantage_lr)

                start = time.perf_counter()
                record = trainer.run_iteration(
                    traversals_per_player=TRAVERSALS_PER_PLAYER
                )
                measured_s += time.perf_counter() - start
                record['measured_training_s'] = measured_s
                record['advantage_learning_rate'] = advantage_lr
                record['strategy_learning_rate'] = 1e-3
                append_jsonl(training_path, [record])
                collect_and_cleanup(evaluator, results_path)

                if measured_s >= next_evaluation_s or measured_s >= total_training_s:
                    validation = trainer.validation_metrics()
                    evaluator.submit(
                        trainer.iteration,
                        trainer.average_policy(),
                        label='learned_average',
                        metadata={
                            'configuration': label,
                            'measured_training_s': measured_s,
                            'advantage_learning_rate': advantage_lr,
                            'advantage_validation_mse': float(np.mean([
                                player['mse'] for player in validation['advantage']
                            ])),
                            'advantage_validation_strategy_tv': float(np.mean([
                                player['strategy_tv'] for player in validation['advantage']
                            ])),
                        },
                    )
                    print(
                        f'[{label}] train={measured_s / 60:.1f}m '
                        f'iter={trainer.iteration} lr={advantage_lr:.2e} '
                        f'pending={evaluator.pending_count}'
                    )
                    while next_evaluation_s <= measured_s:
                        next_evaluation_s += EVALUATE_EVERY_MINUTES * 60.0

            save_policy(trainer.average_policy(), str(policy_dir))
            collect_and_cleanup(evaluator, results_path, wait=True)
        finally:
            evaluator.close(wait=True)

        runs[label] = {
            'run_dir': run_dir,
            'iterations': trainer.iteration,
            'measured_training_s': measured_s,
        }
        del trainer, evaluator
        gc.collect()
        torch.cuda.empty_cache()

print('Finished', len(runs), 'variants')

In [ ]:
summary_rows = []
all_exact = []
all_training = []

for label, run in runs.items():
    training = pd.DataFrame(load_jsonl(run['run_dir'] / 'training.jsonl'))
    exact = pd.DataFrame(load_jsonl(run['run_dir'] / 'exact_results.jsonl'))
    exact = exact.sort_values('measured_training_s').drop_duplicates('iter', keep='last')
    timing = pd.DataFrame(training['timing'].tolist())
    training['configuration'] = label
    exact['configuration'] = label
    all_training.append(training)
    all_exact.append(exact)
    summary_rows.append({
        'configuration': label,
        'iterations completed': int(training['iteration'].iloc[-1]),
        'final exploitability': exact['exploitability'].iloc[-1],
        'best exploitability': exact['exploitability'].min(),
        'normalized AUC': normalized_auc(exact),
        'final advantage MSE': exact['advantage_validation_mse'].iloc[-1],
        'final strategy TV': exact['advantage_validation_strategy_tv'].iloc[-1],
        'final advantage LR': training['advantage_learning_rate'].iloc[-1],
        'mean traversal s': timing['traversal_s'].mean(),
        'mean advantage fit s': timing['advantage_training_s'].mean(),
        'mean strategy fit s': timing['strategy_training_s'].mean(),
    })

summary = pd.DataFrame(summary_rows).set_index('configuration')
exact_df = pd.concat(all_exact, ignore_index=True)
training_df = pd.concat(all_training, ignore_index=True)
display(summary.style.format(precision=6))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for label, group in exact_df.groupby('configuration'):
    group = group.sort_values('measured_training_s')
    x = group['measured_training_s'] / 60.0
    axes[0, 0].plot(x, group['exploitability'], marker='o', label=label)
    axes[0, 1].plot(x, group['advantage_validation_mse'], marker='o', label=label)
    axes[1, 0].plot(
        x,
        group['advantage_validation_strategy_tv'],
        marker='o',
        label=label,
    )

for label, group in training_df.groupby('configuration'):
    group = group.sort_values('measured_training_s')
    axes[1, 1].plot(
        group['measured_training_s'] / 60.0,
        group['advantage_learning_rate'],
        label=label,
    )

axes[0, 0].set_title('Learned-average exact exploitability')
axes[0, 0].set_yscale('log')
axes[0, 1].set_title('Held-out advantage MSE')
axes[1, 0].set_title('Held-out induced-strategy TV')
axes[1, 1].set_title('Advantage learning-rate schedule')
axes[1, 1].set_yscale('log')

for ax in axes.flat:
    ax.set_xlabel('Measured GPU-training minutes')
    ax.grid(alpha=0.3)
    ax.legend()
axes[0, 0].set_ylabel('Exploitability')
axes[0, 1].set_ylabel('MSE')
axes[1, 0].set_ylabel('TV')
axes[1, 1].set_ylabel('Learning rate')
plt.tight_layout()
plt.show()